In [13]:
import geopandas as gpd
import numpy as np
import os, re
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split, RepeatedKFold, cross_validate
from sklearn.metrics import make_scorer, mean_squared_error
import pandas as pd

# === 参数和工具提前定义好 ===
grid_folder = r'D:\seoul\grids\lst_map'
# 这里手动指定代表性文件
filename = 'city2020_lst_ratio_grid_450m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp'
file_path = os.path.join(grid_folder, filename)
gdf_orig = gpd.read_file(file_path)

target_vars = ['nor_2020', 'ext_2020', 'hr_2020']
explanatory = ['BCR(%)','BHV','NDVI','SVF','EV(m)','Dist_BP','Dist_MT','Dist_WB','WR(%)']

# 交叉验证策略与评分
cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=0)
def rmse_scorer(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))
scoring = {'R2':'r2', 'RMSE': make_scorer(rmse_scorer, greater_is_better=False)}

# 模型字典
models = {
    'GBDT': GradientBoostingRegressor(
        n_estimators=4168, learning_rate=0.018,
        max_depth=13, subsample=0.839, random_state=0
    ),
    'RF': RandomForestRegressor(
        n_estimators=983, bootstrap=False,
        max_features=0.692, min_samples_split=2,
        min_samples_leaf=1, random_state=0
    ),
    'ANN': MLPRegressor(
        hidden_layer_sizes=(150,100), activation='relu',
        solver='adam', learning_rate_init=0.01, alpha=1e-3,
        batch_size=64, max_iter=1000, early_stopping=True,
        n_iter_no_change=20, tol=1e-4, random_state=0
    ),
    'AdaBoost': AdaBoostRegressor(
        n_estimators=200, learning_rate=0.5, random_state=0
    )
}

# 用来存放所有 target 和模型的评估结果
records = []

for target in target_vars:
    # 复制一份，以免 dropna 改变原始 gdf
    gdf = gdf_orig.copy()
    gdf = gdf.replace([np.inf, -np.inf], np.nan)\
             .dropna(subset=[target] + explanatory)
    X = gdf[explanatory]
    y = gdf[target]

    # 随机抽 30%
    X, _, y, _ = train_test_split(X, y, train_size=0.3, random_state=0)

    for name, mdl in models.items():
        cv_res = cross_validate(
            mdl, X, y,
            cv=cv,
            scoring=scoring,
            return_train_score=True,
            n_jobs=-1
        )
        # 收集训练/验证的 mean 和 std
        rec = {
            'Target': target,
            'Model': name,
            'Train R2 mean':  np.mean(cv_res['train_R2']),
            'Train R2 std':   np.std(cv_res['train_R2']),
            'Test R2 mean':   np.mean(cv_res['test_R2']),
            'Test R2 std':    np.std(cv_res['test_R2']),
            'Train RMSE mean': -np.mean(cv_res['train_RMSE']),  # 注意符号
            'Train RMSE std':  np.std(cv_res['train_RMSE']),
            'Test RMSE mean':  -np.mean(cv_res['test_RMSE']),
            'Test RMSE std':   np.std(cv_res['test_RMSE'])
        }
        records.append(rec)

# 最后一次性输出到 Excel 或 CSV
df_all = pd.DataFrame(records)
out_path = os.path.join(grid_folder, 'model_comparison_all_targets.xlsx')
df_all.to_excel(out_path, index=False)
print(f"✅ 所有 target 的对比结果已保存到：{out_path}")


✅ 所有 target 的对比结果已保存到：D:\seoul\grids\lst_map\model_comparison_all_targets.xlsx


In [14]:
df_all

,Target,Model,Train R2 mean,Train R2 std,Test R2 mean,Test R2 std,Train RMSE mean,Train RMSE std,Test RMSE mean,Test RMSE std
0,nor_2020,GBDT,1.000000,0.000000e+00,0.870910,0.023364,1.455238e-08,1.537952e-11,1.180784,0.096581
1,nor_2020,RF,1.000000,0.000000e+00,0.872684,0.024211,-0.000000e+00,0.000000e+00,1.172115,0.096419
2,nor_2020,ANN,-1.281894,1.612670e+00,-1.782553,1.564222,4.704463e+00,1.736756e+00,5.239901,1.562528
3,nor_2020,AdaBoost,0.887583,2.789457e-03,0.848236,0.014965,1.111850e+00,1.073735e-02,1.284801,0.071934
4,ext_2020,GBDT,1.000000,0.000000e+00,0.919911,0.011076,1.453815e-08,1.846321e-11,1.264634,0.073058
5,ext_2020,RF,1.000000,0.000000e+00,0.923784,0.013710,-0.000000e+00,0.000000e+00,1.231753,0.092615
6,ext_2020,ANN,-0.922478,1.078845e+00,-1.266041,0.846897,5.971244e+00,1.787061e+00,6.618890,1.380571
7,ext_2020,AdaBoost,0.920073,3.611208e-03,0.891719,0.013957,1.271987e+00,2.698055e-02,1.471285,0.089335
8,hr_2020,GBDT,1.000000,1.110223e-16,0.686420,0.034018,1.456402e-08,1.901730e-11,0.856283,0.052457
9,hr_2020,RF,1.000000,0.000000e+00,0.689359,0.026813,2.450742e-17,2.175198e-18,0.852428,0.038078
